In [ ]:
import sys, os
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df, convert_col
import pickle
import cudf

In [ ]:
%%RecordEventWithColumnInfo
from pathlib import Path
import os
from utils.benchmarks import BENCHMARKS_TO_PATHS
if "IREWR_WITH_MODIN" in os.environ and os.environ["IREWR_WITH_MODIN"] == "True":
    import os
    os.environ["MODIN_ENGINE"] = "ray"
    import ray
    ray.init(num_cpus=int(os.environ['MODIN_CPUS']), runtime_env={'env_vars': {'__MODIN_AUTOIMPORT_PANDAS__': '1'}})
    import modin.pandas as pd
else:
    import pandas as pd
import seaborn as sns 
import matplotlib.pyplot as plt 
import time
import cupy as cp

In [ ]:
%%RecordEventWithColumnInfo
### cell 0 ###

benchmark_name = "creating-player-stats-using-tracking-data"
factor = 0.1
data = pd.read_csv(Path(BENCHMARKS_TO_PATHS[benchmark_name]).parent / "input" / "nfl-big-data-bowl-2023" / "week1.csv")
scout = pd.read_csv(Path(BENCHMARKS_TO_PATHS[benchmark_name]).parent / "input" / "nfl-big-data-bowl-2023" / "pffScoutingData.csv")
plays = pd.read_csv(Path(BENCHMARKS_TO_PATHS[benchmark_name]).parent / "input" / "nfl-big-data-bowl-2023" / "plays.csv")
players = pd.read_csv(Path(BENCHMARKS_TO_PATHS[benchmark_name]).parent / "input" / "nfl-big-data-bowl-2023" / "players.csv")

# Let's merge these data into one set 
data = data.merge(scout, how='left')
data = data.sample(frac=factor, random_state=0)
data.shape

In [ ]:
%%RecordEventWithColumnInfo
### cell 1 ###

data.info()

In [ ]:
%%RecordEventWithColumnInfo
### cell 2 ###

# get ball snap indicies 
idxs = (data
         .loc[data['event']=='ball_snap', 
              'frameId']
         .index
         .values)

# to get 500ms of movement after snap, get 5 rows (each row is 100ms of info)
x = [(idxs+x).tolist() for x in range(0,6)]
idxs = [item for sublist in x for item in sublist] #the output x is a list of lists, so this is just to flatten the list

# filter for snap + 500ms of data only using our selected indicies
df = data.reindex(idxs)

In [ ]:
%%RecordEventWithColumnInfo
### cell 3 ###

gid = 2021090900
pid = 97 
nid = 25511 
df.loc[(df['gameId']==gid) & (df['playId']==pid) & (df['nflId']==nid)]

In [ ]:
%%RecordEventWithColumnInfo
### cell 4 ###

# get line of scrimmage info to compute block/rush depth relative to LOS
los = (data
        .loc[(data['team']=='football') & 
             (data['frameId']==1), 
             ['gameId', 'playId', 'x']]
        .rename(columns={'x':'los'}))

# merge LOS info back to subsetted data
df = df.merge(los)

In [ ]:
%%RecordEventWithColumnInfo
### cell 5 ###

df.loc[(df['gameId']==gid) & (df['playId']==pid) & (df['nflId']==nid)]

In [ ]:
%%RecordEventWithColumnInfo
### cell 6 ###

# get difference from LOS for all frames and players 
df['los_diff'] = df['x'].sub(df['los'])

# multiply by -1 for plays going the "left" direction 
# this is so pass block is monotonic in the same direction (as well as pass rush)
df.loc[df['playDirection']=='left', 'los_diff'] = df.loc[df['playDirection']=='left', 'los_diff'].mul(-1)

# merge onto play info to get possession team (could do this anywhere, i do it here for no real optimal reason)
df = plays.loc[:, ['gameId', 'playId', 'possessionTeam']].merge(df)

In [ ]:
%%RecordEventWithColumnInfo
### cell 7 ###

df.loc[(df['gameId']==gid) & (df['playId']==pid) & (df['nflId']==nid)]

In [ ]:
%%RecordEventWithColumnInfo
### cell 8 ###

# indicate if a player is on the possession team (1), the defensive team (0), or neither aka the football (-1)
df['posTeam'] = 0
df.loc[df['possessionTeam']==df['team'], 'posTeam'] = 1 
df.loc[df['team']=='football', 'posTeam'] = -1

# create initial snap speed dataframe (mean of speed and acceleration per player)
snap_speed = (df
              .loc[:, ['nflId','s','a']]
              .groupby('nflId', 
                       as_index=False)
              .mean())

In [ ]:
%%RecordEventWithColumnInfo
### cell 9 ###

snap_speed.head()

In [ ]:
%%RecordEventWithColumnInfo
### cell 10 ###

# given whether a offense player or defense player, aggregate by maxmimum or minimum LOS difference, respectively. 
# e.g. if o-lineman has more a negative LOS diff, they allow more pass rush penetration 
off = df.loc[df['posTeam']==1, ['gameId', 'playId', 'nflId', 'los_diff']].groupby(['gameId', 'playId', 'nflId'], as_index=False).max()
def1 = df.loc[df['posTeam']==0, ['gameId', 'playId', 'nflId', 'los_diff']].groupby(['gameId', 'playId', 'nflId'], as_index=False).min()
los_diff = off._append(def1)
los_diff = (los_diff
            .loc[:, ['nflId', 'los_diff']]
            .groupby('nflId', 
                     as_index=False)
            .mean())

# merge LOS diff data back onto snap speed
snap_speed = snap_speed.merge(los_diff)
snap_speed = snap_speed.rename(columns={'s':'snap_s', 'a':'snap_a', 'los_diff':'snap_los_diff'})

In [ ]:
%%RecordEventWithColumnInfo
### cell 11 ###

snap_speed.head()

In [ ]:
%%RecordEventWithColumnInfo
### cell 12 ###

df_plt = players.loc[:, ['nflId', 'officialPosition', 'displayName']].merge(snap_speed)

In [ ]:
%%RecordEventWithColumnInfo
### cell 13 ###

df_plt.loc[(df_plt['officialPosition']=='DE') & (df_plt['snap_los_diff']<0)].sort_values('snap_los_diff')